# Probability and Statistics Foundations — Collaborative Notebook

**Session:** Descriptive statistics, probability distributions, probability rules & Bayes' theorem, correlation
**Duration:** 4 hours
**Goal:** Rebuild statistical and probabilistic fluency as mathematical foundations for Module 1's algorithms.

> This notebook teaches the **math**, not its application in ML evaluation or model selection (IIT Kharagpur faculty cover that), and not hypothesis testing / confidence intervals (see the asynchronous revision kit).


In [ ]:
# Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Environment ready:", np.__version__, pd.__version__)

---
# Part 1 — Descriptive Statistics (25 min)

## 1.1 Mean, median, mode

In [ ]:
data = np.array([4, 8, 15, 16, 23, 42, 8, 8, 15])

mean_val = np.mean(data)
median_val = np.median(data)
mode_val = stats.mode(data, keepdims=True)

print("Mean:", mean_val)
print("Median:", median_val)
print("Mode:", mode_val.mode[0], "(count:", mode_val.count[0], ")")

**When each is meaningful:**
- **Mean** — best for symmetric data without extreme outliers
- **Median** — robust to outliers and skew (e.g., income data)
- **Mode** — the only sensible "average" for categorical data

## 1.2 Variance and standard deviation

In [ ]:
variance = np.var(data)
std_dev = np.std(data)

print("Variance:", variance)
print("Standard deviation:", std_dev)
# Standard deviation is in the same units as the data — easier to interpret than variance

## 1.3 Percentiles and interquartile range

In [ ]:
p25 = np.percentile(data, 25)
p50 = np.percentile(data, 50)
p75 = np.percentile(data, 75)
iqr = p75 - p25

print(f"25th percentile: {p25}")
print(f"50th percentile (median): {p50}")
print(f"75th percentile: {p75}")
print(f"IQR: {iqr}")

## 1.4 Computing all of the above in NumPy and Pandas

In [ ]:
df = pd.read_csv("students_survey.csv")
df[["study_hours_per_week", "exam_score"]].describe()

In [ ]:
# Individual stats via Pandas
print("Mean exam score:", df["exam_score"].mean())
print("Median exam score:", df["exam_score"].median())
print("Std dev exam score:", df["exam_score"].std())
print("IQR exam score:", df["exam_score"].quantile(0.75) - df["exam_score"].quantile(0.25))

**Try it:** Compute the mean, standard deviation, and IQR of `attendance_pct`.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
mean_att = df["attendance_pct"].mean()
std_att = df["attendance_pct"].std()
iqr_att = df["attendance_pct"].quantile(0.75) - df["attendance_pct"].quantile(0.25)
```
</details>

---
# Part 2 — Probability Distributions (35 min)

A **probability distribution** maps every possible outcome of a random variable to its probability (or probability density).

## 2.1 Normal (Gaussian) distribution

Parameters: mean (μ) and standard deviation (σ). The **68-95-99.7 rule**: about 68% of values fall within 1σ of the mean, 95% within 2σ, 99.7% within 3σ.

In [ ]:
mu, sigma = 0, 1
x = np.linspace(-4, 4, 400)
pdf = stats.norm.pdf(x, mu, sigma)

plt.figure(figsize=(7, 4))
plt.plot(x, pdf, color="steelblue")
for k, alpha in zip([1, 2, 3], [0.35, 0.2, 0.1]):
    plt.fill_between(x, pdf, where=(np.abs(x) <= k), color="steelblue", alpha=alpha)
plt.title("Standard Normal Distribution — 68-95-99.7 rule")
plt.xlabel("z-score")
plt.ylabel("Density")
plt.show()

## 2.2 Bernoulli distribution — a single binary trial

In [ ]:
p = 0.3  # probability of success
outcomes = [0, 1]
probs = [1 - p, p]

plt.figure(figsize=(5, 4))
plt.bar(outcomes, probs, color=["salmon", "steelblue"], tick_label=["Failure (0)", "Success (1)"])
plt.title(f"Bernoulli(p={p})")
plt.ylabel("Probability")
plt.show()

## 2.3 Binomial distribution — multiple binary trials

In [ ]:
n_trials, p = 10, 0.3
k = np.arange(0, n_trials + 1)
pmf = stats.binom.pmf(k, n_trials, p)

plt.figure(figsize=(7, 4))
plt.bar(k, pmf, color="steelblue")
plt.title(f"Binomial(n={n_trials}, p={p})")
plt.xlabel("Number of successes")
plt.ylabel("Probability")
plt.show()

## 2.4 Sampling from distributions with np.random

In [ ]:
normal_sample = np.random.normal(loc=70, scale=10, size=1000)
binomial_sample = np.random.binomial(n=10, p=0.3, size=1000)
bernoulli_sample = np.random.binomial(n=1, p=0.3, size=1000)  # Bernoulli = Binomial with n=1

print("Sample mean (normal):", normal_sample.mean().round(2))
print("Sample mean (binomial):", binomial_sample.mean().round(2))
print("Sample mean (bernoulli):", bernoulli_sample.mean().round(2))

**Try it:** Plot a histogram of `exam_score` and, by eye, decide which distribution shape it resembles most.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
plt.figure(figsize=(6, 4))
plt.hist(df["exam_score"], bins=20, color="steelblue")
plt.title("Distribution of Exam Score")
plt.xlabel("Exam score")
plt.ylabel("Count")
plt.show()
# Roughly bell-shaped / approximately normal
```
</details>

---
# Part 3 — Probability Rules and Bayes' Theorem (50 min)

## 3.1 Joint and marginal probability

In [ ]:
joint_counts = pd.crosstab(df["study_group"], df["passed"])
joint_counts

In [ ]:
# Joint probability table: P(study_group AND passed)
joint_prob = joint_counts / joint_counts.values.sum()
joint_prob

In [ ]:
# Marginal probability: sum across rows / columns of the joint table
marginal_group = joint_prob.sum(axis=1)   # P(study_group)
marginal_passed = joint_prob.sum(axis=0)  # P(passed)

print("P(study_group):\n", marginal_group)
print("\nP(passed):\n", marginal_passed)

## 3.2 Conditional probability

$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)}$$

**Worked example:** What is P(passed = Yes | study_group = C)?

In [ ]:
p_c_and_yes = joint_prob.loc["C", "Yes"]
p_c = marginal_group["C"]
p_yes_given_c = p_c_and_yes / p_c

print(f"P(passed=Yes and group=C) = {p_c_and_yes:.3f}")
print(f"P(group=C) = {p_c:.3f}")
print(f"P(passed=Yes | group=C) = {p_yes_given_c:.3f}")

## 3.3 Independence

Two events are independent when P(A | B) = P(A) — knowing B happened tells you nothing new about A.

In [ ]:
p_yes = marginal_passed["Yes"]
print(f"P(passed=Yes) = {p_yes:.3f}")
print(f"P(passed=Yes | group=C) = {p_yes_given_c:.3f}")
print("Equal?" , np.isclose(p_yes, p_yes_given_c, atol=0.01))
# They differ noticeably here — study_group and passing are NOT independent

## 3.4 Bayes' theorem — derivation

Starting from the definition of conditional probability:

$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)} \quad \text{and} \quad P(B \mid A) = \frac{P(A \text{ and } B)}{P(A)}$$

Both equal $P(A \text{ and } B)$, so $P(A\mid B)\,P(B) = P(B\mid A)\,P(A)$. Dividing by $P(B)$:

$$P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}$$

This is **Bayes' theorem**.

## 3.5 Worked example 1 (simple)

A fair coin is flipped. Given it landed heads, what's the probability it was flipped by a specific person who flips heads 70% of the time, if we already know the flipper is that person with prior probability 0.5?

In [ ]:
prior = 0.5           # P(this person is flipping)
likelihood = 0.7       # P(heads | this person)
p_heads_other = 0.5    # P(heads | someone else flipping)
prior_other = 0.5

evidence = likelihood * prior + p_heads_other * prior_other  # P(heads)
posterior = (likelihood * prior) / evidence

print(f"P(this person | heads) = {posterior:.3f}")

## 3.6 Worked example 2 (medium)

Using the dataset: given a student **passed**, what's the probability they were in **study_group C**? Use Bayes' theorem, not just the joint table, to confirm the two approaches agree.

In [ ]:
prior_c = marginal_group["C"]                 # P(group=C)
likelihood_pass_given_c = p_yes_given_c        # P(passed=Yes | group=C), from 3.2
evidence_pass = marginal_passed["Yes"]         # P(passed=Yes)

posterior_c_given_pass = (likelihood_pass_given_c * prior_c) / evidence_pass
print(f"P(group=C | passed=Yes) via Bayes' theorem: {posterior_c_given_pass:.3f}")

# Cross-check directly from the joint table
direct = joint_prob.loc["C", "Yes"] / marginal_passed["Yes"]
print(f"P(group=C | passed=Yes) direct from table:  {direct:.3f}")

## 3.7 Worked example 3 (increasing complexity)

A classic diagnostic-test style problem: 2% of students are genuinely at risk of failing ("at-risk"). A screening quiz correctly flags 90% of at-risk students, but also incorrectly flags 15% of not-at-risk students. If a student is flagged, what's the probability they are actually at-risk?

In [ ]:
p_at_risk = 0.02
p_flag_given_at_risk = 0.90
p_flag_given_not_at_risk = 0.15
p_not_at_risk = 1 - p_at_risk

p_flag = (p_flag_given_at_risk * p_at_risk) + (p_flag_given_not_at_risk * p_not_at_risk)
p_at_risk_given_flag = (p_flag_given_at_risk * p_at_risk) / p_flag

print(f"P(flagged) = {p_flag:.3f}")
print(f"P(at-risk | flagged) = {p_at_risk_given_flag:.3f}")
# Even with a 90%-sensitive test, a low base rate keeps this posterior surprisingly low

## 3.8 Vocabulary: prior, likelihood, posterior

$$\underbrace{P(A \mid B)}_{\text{posterior}} = \frac{\overbrace{P(B \mid A)}^{\text{likelihood}} \; \overbrace{P(A)}^{\text{prior}}}{P(B)}$$

- **Prior** — P(A), what you believed before seeing the evidence
- **Likelihood** — P(B | A), how probable the evidence is if A is true
- **Posterior** — P(A | B), your updated belief after seeing the evidence

*(Mathematical definitions only — using these in model evaluation is covered separately.)*

**Try it:** Using the joint table, compute P(study_group = A | passed = No) with Bayes' theorem, and cross-check against the direct calculation.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
prior_a = marginal_group["A"]
likelihood_no_given_a = joint_prob.loc["A", "No"] / prior_a
evidence_no = marginal_passed["No"]

posterior_a_given_no = (likelihood_no_given_a * prior_a) / evidence_no
direct = joint_prob.loc["A", "No"] / marginal_passed["No"]
print(posterior_a_given_no, direct)
```
</details>

---
# Part 4 — Correlation (20 min)

## 4.1 Pearson correlation coefficient

In [ ]:
pearson_r = df["study_hours_per_week"].corr(df["exam_score"], method="pearson")
print(f"Pearson r (study hours vs. exam score): {pearson_r:.3f}")
# Range: [-1, 1]. Close to +1 = strong positive linear relationship.

## 4.2 Spearman's rank correlation — when Pearson fails

In [ ]:
pearson_conf = df["exam_score"].corr(df["confidence_rating"], method="pearson")
spearman_conf = df["exam_score"].corr(df["confidence_rating"], method="spearman")

print(f"Pearson r:  {pearson_conf:.3f}")
print(f"Spearman rho: {spearman_conf:.3f}")
# confidence_rating rises with exam_score but levels off (non-linear, monotonic) —
# Spearman (rank-based) captures that relationship more faithfully than Pearson.

## 4.3 Computing correlation in Pandas: .corr()

In [ ]:
numeric_cols = ["study_hours_per_week", "sleep_hours_per_night", "attendance_pct", "confidence_rating", "exam_score"]
corr_matrix = df[numeric_cols].corr()  # defaults to Pearson

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 4.4 Why correlation does not imply causation

**Example 1:** Ice cream sales and drowning incidents are positively correlated — both rise in summer. Heat is the confounding variable; ice cream doesn't cause drowning.

**Example 2:** In this dataset, `attendance_pct` and `exam_score` may correlate — but a highly motivated student could cause *both* high attendance and a high score, without attendance itself being the direct cause.

---
# Part 5 — Hands-On Exercise (40 min)

Work through each step using `students_survey.csv`. Solutions are collapsed under each step — try first, then check.


### Step 1 — Descriptive statistics
Compute mean, median, standard deviation, and IQR for `study_hours_per_week`, `attendance_pct`, and `exam_score`.

In [ ]:
# TODO: Step 1


<details><summary>Solution</summary>

```python
data = pd.read_csv("students_survey.csv")
cols = ["study_hours_per_week", "attendance_pct", "exam_score"]
for c in cols:
    print(c, "mean:", data[c].mean(), "median:", data[c].median(),
          "std:", data[c].std(),
          "IQR:", data[c].quantile(0.75) - data[c].quantile(0.25))
```
</details>

### Step 2 — Identify distribution shapes
Plot histograms of `exam_score` and `confidence_rating`. Which looks closer to Normal? Which looks skewed?

In [ ]:
# TODO: Step 2


<details><summary>Solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(data["exam_score"], bins=20, color="steelblue")
axes[0].set_title("Exam Score")
axes[1].hist(data["confidence_rating"], bins=20, color="darkorange")
axes[1].set_title("Confidence Rating")
plt.show()
# exam_score: roughly Normal / bell-shaped
# confidence_rating: left-skewed (bunched toward higher values)
```
</details>

### Step 3 — Build a joint probability table
Build a joint probability table between `study_mode` and `passed`.

In [ ]:
# TODO: Step 3


<details><summary>Solution</summary>

```python
joint = pd.crosstab(data["study_mode"], data["passed"])
joint_p = joint / joint.values.sum()
joint_p
```
</details>

### Step 4 — Compute a conditional probability
From your table, compute P(passed = Yes | study_mode = Instructor-led).

In [ ]:
# TODO: Step 4


<details><summary>Solution</summary>

```python
p_led = joint_p.loc["Instructor-led"].sum()
p_led_and_yes = joint_p.loc["Instructor-led", "Yes"]
p_yes_given_led = p_led_and_yes / p_led
p_yes_given_led
```
</details>

### Step 5 — Apply Bayes' theorem
A student passed. Using Bayes' theorem, what's the probability they were in the Instructor-led mode? Cross-check against the direct calculation from the table.

In [ ]:
# TODO: Step 5


<details><summary>Solution</summary>

```python
prior_led = joint_p.loc["Instructor-led"].sum()
likelihood_yes_given_led = p_yes_given_led
evidence_yes = joint_p["Yes"].sum()

posterior_led_given_yes = (likelihood_yes_given_led * prior_led) / evidence_yes
direct = joint_p.loc["Instructor-led", "Yes"] / evidence_yes
posterior_led_given_yes, direct
```
</details>

### Step 6 — Correlation matrix
Compute and interpret the correlation matrix of the numeric columns. Which pair is most strongly correlated?

In [ ]:
# TODO: Step 6


<details><summary>Solution</summary>

```python
numeric_cols = ["study_hours_per_week", "sleep_hours_per_night", "attendance_pct", "confidence_rating", "exam_score"]
corr = data[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()
# study_hours_per_week and exam_score are typically the strongest pair
```
</details>

---
## Wrap-up

You've now practiced:
- **Descriptive statistics**: mean, median, mode, variance, std dev, percentiles, IQR
- **Probability distributions**: Normal, Bernoulli, Binomial — visualizing and sampling
- **Probability rules & Bayes' theorem**: joint, marginal, conditional probability, independence, and three worked Bayes examples
- **Correlation**: Pearson vs. Spearman, computing with `.corr()`, and why correlation isn't causation

**Not covered here:** ML evaluation metrics, bias-variance tradeoff, cross-validation, model selection (IIT Kharagpur faculty), hypothesis testing and confidence intervals (asynchronous revision kit).